# Data Exploration & Analysis
Comprehensive exploration of the Healthcare Intelligence System synthetic dataset.

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# PySpark imports
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import *

# Data analysis & visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Project imports
from src.data_processing.spark_session import create_spark_session

# Settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', 50)
print("All imports successful!")

In [ ]:
# Create Spark session
spark = create_spark_session(app_name="HealthcareEDA", master="local[*]")
print(f"Spark version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

In [ ]:
# Load all datasets using HealthcareDataLoader
from src.data_processing.data_loader import HealthcareDataLoader

loader = HealthcareDataLoader(spark, data_dir=os.path.join(project_root, 'data'))

# Load each dataset
patients_df = loader.load_patients()
vitals_df = loader.load_vitals()
symptoms_df = loader.load_symptoms()
lab_results_df = loader.load_lab_results()
clinical_notes_df = loader.load_clinical_notes()
ground_truth_df = loader.load_ground_truth()

# Validate each dataset
datasets = {
    'patients': patients_df,
    'vitals': vitals_df,
    'symptoms': symptoms_df,
    'lab_results': lab_results_df,
    'clinical_notes': clinical_notes_df,
    'ground_truth': ground_truth_df
}

for name, df in datasets.items():
    validation = loader.validate_data(df, name)
    print(f"\n{'='*50}")
    print(f"Dataset: {name}")
    print(f"  Rows: {df.count()}, Columns: {len(df.columns)}")
    print(f"  Validation: {'PASSED' if validation['is_valid'] else 'FAILED'}")
    if not validation['is_valid']:
        for issue in validation.get('issues', []):
            print(f"    - {issue}")

## Patient Demographics Analysis

In [ ]:
# Convert patients to pandas and merge with ground truth for diagnosis labels
patients_pd = patients_df.toPandas()
ground_truth_pd = ground_truth_df.toPandas()
patients_merged = patients_pd.merge(ground_truth_pd[['patient_id', 'diagnosis', 'risk_level']], on='patient_id', how='left')

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Age distribution histogram
axes[0].hist(patients_merged['age'], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_title('Age Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].axvline(patients_merged['age'].mean(), color='red', linestyle='--', label=f"Mean: {patients_merged['age'].mean():.1f}")
axes[0].legend()

# Sex distribution pie chart
sex_counts = patients_merged['sex'].value_counts()
axes[1].pie(sex_counts.values, labels=sex_counts.index, autopct='%1.1f%%',
            colors=['#66b3ff', '#ff9999', '#99ff99'], startangle=90)
axes[1].set_title('Sex Distribution', fontsize=14, fontweight='bold')

# BMI distribution by diagnosis (violin plot)
diagnoses = patients_merged['diagnosis'].dropna().unique()
bmi_data = [patients_merged[patients_merged['diagnosis'] == d]['bmi'].dropna().values for d in diagnoses]
parts = axes[2].violinplot(bmi_data, showmeans=True, showmedians=True)
axes[2].set_xticks(range(1, len(diagnoses) + 1))
axes[2].set_xticklabels(diagnoses, rotation=45, ha='right', fontsize=9)
axes[2].set_title('BMI Distribution by Diagnosis', fontsize=14, fontweight='bold')
axes[2].set_ylabel('BMI')

plt.tight_layout()
plt.savefig(os.path.join(project_root, 'outputs', 'patient_demographics.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPatient Demographics Summary:")
print(f"  Total patients: {len(patients_merged)}")
print(f"  Age range: {patients_merged['age'].min()} - {patients_merged['age'].max()}")
print(f"  Mean BMI: {patients_merged['bmi'].mean():.2f}")

## Vital Signs Analysis

In [ ]:
# Vital signs analysis - box plots grouped by diagnosis
vitals_pd = vitals_df.toPandas()
vitals_merged = vitals_pd.merge(ground_truth_pd[['patient_id', 'diagnosis']], on='patient_id', how='left')

vital_cols = ['heart_rate', 'systolic_bp', 'temperature', 'oxygen_saturation']
vital_titles = ['Heart Rate (bpm)', 'Systolic BP (mmHg)', 'Temperature (°F)', 'Oxygen Saturation (%)']

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
axes = axes.flatten()

for idx, (col, title) in enumerate(zip(vital_cols, vital_titles)):
    if col in vitals_merged.columns:
        data_groups = []
        labels = []
        for diag in sorted(vitals_merged['diagnosis'].dropna().unique()):
            vals = vitals_merged[vitals_merged['diagnosis'] == diag][col].dropna()
            if len(vals) > 0:
                data_groups.append(vals.values)
                labels.append(diag)
        
        bp = axes[idx].boxplot(data_groups, labels=labels, patch_artist=True)
        colors = plt.cm.Set3(np.linspace(0, 1, len(labels)))
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
        
        axes[idx].set_title(title, fontsize=14, fontweight='bold')
        axes[idx].set_ylabel(title.split('(')[0].strip())
        axes[idx].tick_params(axis='x', rotation=45)

plt.suptitle('Vital Signs Distribution by Diagnosis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(project_root, 'outputs', 'vital_signs_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

# Summary statistics
print("\nVital Signs Summary Statistics:")
for col in vital_cols:
    if col in vitals_merged.columns:
        print(f"\n  {col}:")
        print(f"    Mean: {vitals_merged[col].mean():.2f}, Std: {vitals_merged[col].std():.2f}")
        print(f"    Min: {vitals_merged[col].min():.2f}, Max: {vitals_merged[col].max():.2f}")

## Symptom Distribution

In [ ]:
# Symptom prevalence heatmap per diagnosis class
symptoms_pd = symptoms_df.toPandas()
symptoms_merged = symptoms_pd.merge(ground_truth_pd[['patient_id', 'diagnosis']], on='patient_id', how='left')

# Identify symptom columns (binary columns excluding patient_id and diagnosis)
symptom_cols = [col for col in symptoms_merged.columns 
                if col not in ['patient_id', 'diagnosis', 'encounter_id', 'timestamp', 'symptom_text']
                and symptoms_merged[col].dtype in ['int64', 'float64', 'bool']]

# If symptoms are in a single column (long format), pivot them
if 'symptom_name' in symptoms_merged.columns:
    # Long format - create pivot table
    prevalence = symptoms_merged.groupby(['diagnosis', 'symptom_name']).size().unstack(fill_value=0)
    # Convert to percentage
    prevalence = prevalence.div(symptoms_merged.groupby('diagnosis')['patient_id'].nunique(), axis=0) * 100
    prevalence = prevalence.T
elif len(symptom_cols) > 0:
    # Wide format - compute prevalence per diagnosis
    prevalence = symptoms_merged.groupby('diagnosis')[symptom_cols].mean() * 100
    prevalence = prevalence.T
else:
    # Try to extract symptoms from available columns
    symptom_cols = [col for col in symptoms_merged.columns if col not in ['patient_id', 'diagnosis']]
    prevalence = symptoms_merged.groupby('diagnosis')[symptom_cols[:20]].mean() * 100
    prevalence = prevalence.T

# Plot heatmap
fig, ax = plt.subplots(figsize=(14, max(8, len(prevalence) * 0.4)))
sns.heatmap(prevalence, annot=True, fmt='.1f', cmap='YlOrRd', 
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Prevalence (%)'})
ax.set_title('Symptom Prevalence by Diagnosis Class (%)', fontsize=16, fontweight='bold')
ax.set_xlabel('Diagnosis', fontsize=12)
ax.set_ylabel('Symptom', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(project_root, 'outputs', 'symptom_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTotal unique symptoms analyzed: {len(prevalence)}")
print(f"Diagnoses: {list(prevalence.columns)}")

## Lab Results Analysis

In [ ]:
# Lab results analysis
lab_pd = lab_results_df.toPandas()
lab_merged = lab_pd.merge(ground_truth_pd[['patient_id', 'diagnosis']], on='patient_id', how='left')

# Mean value per test per diagnosis - grouped bar chart
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Grouped bar chart of mean lab values per diagnosis
if 'test_name' in lab_merged.columns and 'value' in lab_merged.columns:
    mean_vals = lab_merged.groupby(['diagnosis', 'test_name'])['value'].mean().unstack()
    mean_vals.plot(kind='bar', ax=axes[0], width=0.8)
    axes[0].set_title('Mean Lab Values by Diagnosis', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Diagnosis')
    axes[0].set_ylabel('Mean Value')
    axes[0].legend(title='Test', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    axes[0].tick_params(axis='x', rotation=45)
    
    # Distribution of abnormal lab values per diagnosis
    if 'is_abnormal' in lab_merged.columns:
        abnormal_col = 'is_abnormal'
    elif 'abnormal_flag' in lab_merged.columns:
        abnormal_col = 'abnormal_flag'
    else:
        # Create abnormal flag based on reference range
        abnormal_col = None
    
    if abnormal_col:
        abnormal_counts = lab_merged.groupby('diagnosis')[abnormal_col].mean() * 100
        abnormal_counts.plot(kind='bar', ax=axes[1], color='coral', edgecolor='black')
        axes[1].set_title('Abnormal Lab Values by Diagnosis (%)', fontsize=14, fontweight='bold')
        axes[1].set_xlabel('Diagnosis')
        axes[1].set_ylabel('Abnormal Rate (%)')
        axes[1].tick_params(axis='x', rotation=45)
    else:
        # Fallback: show test count distribution
        test_counts = lab_merged.groupby('diagnosis')['test_name'].count()
        test_counts.plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='black')
        axes[1].set_title('Number of Lab Tests by Diagnosis', fontsize=14, fontweight='bold')
        axes[1].set_xlabel('Diagnosis')
        axes[1].set_ylabel('Count')
        axes[1].tick_params(axis='x', rotation=45)
else:
    # Wide format lab results
    lab_cols = [c for c in lab_merged.columns if c not in ['patient_id', 'diagnosis', 'encounter_id', 'timestamp']]
    lab_means = lab_merged.groupby('diagnosis')[lab_cols[:10]].mean()
    lab_means.plot(kind='bar', ax=axes[0], width=0.8)
    axes[0].set_title('Mean Lab Values by Diagnosis', fontsize=14, fontweight='bold')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(project_root, 'outputs', 'lab_results_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\nLab Results Summary:")
print(f"  Total lab records: {len(lab_merged)}")
if 'test_name' in lab_merged.columns:
    print(f"  Unique tests: {lab_merged['test_name'].nunique()}")
    print(f"  Tests: {list(lab_merged['test_name'].unique())}")

## Clinical Notes Analysis

In [ ]:
# Clinical notes analysis
from collections import Counter
import re

notes_pd = clinical_notes_df.toPandas()
notes_merged = notes_pd.merge(ground_truth_pd[['patient_id', 'diagnosis']], on='patient_id', how='left')

# Determine the text column name
text_col = None
for col_name in ['note_text', 'clinical_note', 'text', 'note', 'notes']:
    if col_name in notes_merged.columns:
        text_col = col_name
        break

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

if text_col:
    # Word count distribution
    notes_merged['word_count'] = notes_merged[text_col].astype(str).apply(lambda x: len(x.split()))
    axes[0].hist(notes_merged['word_count'], bins=30, edgecolor='black', alpha=0.7, color='mediumpurple')
    axes[0].set_title('Word Count Distribution', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Word Count')
    axes[0].set_ylabel('Frequency')
    axes[0].axvline(notes_merged['word_count'].mean(), color='red', linestyle='--',
                    label=f"Mean: {notes_merged['word_count'].mean():.0f}")
    axes[0].legend()

# Note type distribution
type_col = None
for col_name in ['note_type', 'type', 'category']:
    if col_name in notes_merged.columns:
        type_col = col_name
        break

if type_col:
    type_counts = notes_merged[type_col].value_counts()
    axes[1].barh(type_counts.index, type_counts.values, color='teal', edgecolor='black')
    axes[1].set_title('Note Type Distribution', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Count')
else:
    axes[1].text(0.5, 0.5, 'Note type column\nnot found', transform=axes[1].transAxes,
                 ha='center', va='center', fontsize=14)
    axes[1].set_title('Note Type Distribution', fontsize=14, fontweight='bold')

# Top-20 frequent terms per diagnosis (combined view)
if text_col:
    stop_words = {'the', 'a', 'an', 'is', 'was', 'are', 'were', 'be', 'been', 'being',
                  'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could',
                  'should', 'may', 'might', 'shall', 'can', 'to', 'of', 'in', 'for',
                  'on', 'with', 'at', 'by', 'from', 'as', 'into', 'through', 'during',
                  'and', 'but', 'or', 'nor', 'not', 'so', 'yet', 'both', 'either',
                  'neither', 'each', 'every', 'all', 'any', 'few', 'more', 'most',
                  'other', 'some', 'such', 'no', 'only', 'own', 'same', 'than',
                  'too', 'very', 'just', 'because', 'this', 'that', 'these', 'those',
                  'it', 'its', 'he', 'she', 'they', 'them', 'his', 'her', 'their',
                  'patient', 'also', 'which', 'who', 'whom', 'what', 'where', 'when'}
    
    all_text = ' '.join(notes_merged[text_col].astype(str).tolist()).lower()
    words = re.findall(r'[a-z]+', all_text)
    words = [w for w in words if w not in stop_words and len(w) > 2]
    word_counts = Counter(words).most_common(20)
    
    words_list, counts_list = zip(*word_counts) if word_counts else ([], [])
    axes[2].barh(range(len(words_list)), counts_list, color='darkorange', edgecolor='black')
    axes[2].set_yticks(range(len(words_list)))
    axes[2].set_yticklabels(words_list)
    axes[2].invert_yaxis()
    axes[2].set_title('Top 20 Frequent Terms', fontsize=14, fontweight='bold')
    axes[2].set_xlabel('Frequency')

plt.tight_layout()
plt.savefig(os.path.join(project_root, 'outputs', 'clinical_notes_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\nClinical Notes Summary:")
print(f"  Total notes: {len(notes_merged)}")
if text_col:
    print(f"  Avg word count: {notes_merged['word_count'].mean():.1f}")
    print(f"  Max word count: {notes_merged['word_count'].max()}")

## Class Distribution & Balance

In [ ]:
# Class distribution analysis
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Diagnosis distribution bar chart
diag_counts = ground_truth_pd['diagnosis'].value_counts()
colors_diag = plt.cm.Set2(np.linspace(0, 1, len(diag_counts)))
bars = axes[0].bar(range(len(diag_counts)), diag_counts.values, color=colors_diag, edgecolor='black')
axes[0].set_xticks(range(len(diag_counts)))
axes[0].set_xticklabels(diag_counts.index, rotation=45, ha='right')
axes[0].set_title('Diagnosis Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Diagnosis')
axes[0].set_ylabel('Count')

# Add count labels on bars
for bar, count in zip(bars, diag_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                 str(count), ha='center', va='bottom', fontweight='bold')

# Risk level distribution
if 'risk_level' in ground_truth_pd.columns:
    risk_order = ['low', 'moderate', 'high', 'critical']
    risk_counts = ground_truth_pd['risk_level'].value_counts()
    # Reorder if possible
    ordered_risks = [r for r in risk_order if r in risk_counts.index]
    remaining = [r for r in risk_counts.index if r not in ordered_risks]
    ordered_risks.extend(remaining)
    risk_counts = risk_counts.reindex(ordered_risks)
    
    risk_colors = {'low': '#2ecc71', 'moderate': '#f39c12', 'high': '#e74c3c', 'critical': '#8e44ad'}
    bar_colors = [risk_colors.get(r, '#95a5a6') for r in risk_counts.index]
    
    bars2 = axes[1].bar(range(len(risk_counts)), risk_counts.values, color=bar_colors, edgecolor='black')
    axes[1].set_xticks(range(len(risk_counts)))
    axes[1].set_xticklabels(risk_counts.index, rotation=45, ha='right')
    axes[1].set_title('Risk Level Distribution', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Risk Level')
    axes[1].set_ylabel('Count')
    
    for bar, count in zip(bars2, risk_counts.values):
        axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                     str(count), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(project_root, 'outputs', 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

# Class balance metrics
print("\nClass Distribution Summary:")
print(f"  Total samples: {len(ground_truth_pd)}")
print(f"  Number of classes: {ground_truth_pd['diagnosis'].nunique()}")
print(f"\n  Diagnosis counts:")
for diag, count in diag_counts.items():
    print(f"    {diag}: {count} ({count/len(ground_truth_pd)*100:.1f}%)")

imbalance_ratio = diag_counts.max() / diag_counts.min()
print(f"\n  Class imbalance ratio (max/min): {imbalance_ratio:.2f}")

## Correlation Analysis

In [ ]:
# Correlation matrix heatmap of numeric patient features
numeric_cols = patients_merged.select_dtypes(include=[np.number]).columns.tolist()
# Remove non-informative columns
exclude_cols = ['patient_id']
numeric_cols = [c for c in numeric_cols if c not in exclude_cols]

if len(numeric_cols) > 1:
    corr_matrix = patients_merged[numeric_cols].corr()
    
    fig, ax = plt.subplots(figsize=(14, 12))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, square=True, linewidths=0.5, ax=ax,
                cbar_kws={'label': 'Correlation Coefficient'})
    ax.set_title('Correlation Matrix - Patient Numeric Features', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(project_root, 'outputs', 'correlation_matrix.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    # Find strongest correlations
    print("\nTop 10 Strongest Correlations (absolute value):")
    corr_pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], 
                             corr_matrix.iloc[i, j]))
    corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
    for feat1, feat2, corr in corr_pairs[:10]:
        print(f"  {feat1} <-> {feat2}: {corr:.3f}")
else:
    print("Insufficient numeric columns for correlation analysis.")

In [ ]:
# Stop Spark session
spark.stop()
print("Spark session stopped. Data exploration complete!")